In [22]:
"""
Imports variables from urban_data_variables module and defines scraped files variables .
"""
import cyrtranslit
import pandas as pd
import geopandas as gpd
import numpy as np
import re
from variables_functions import (
    district_codes,
    neighborhoods,
    to_latin_bg
)

sofia_for_rent_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_rent_scraped.csv')
sofia_for_sale_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_sale_scraped.csv')

pd.set_option('display.max_rows', None)


In [ ]:
def standardize_locations(name):
    """
    Standardizes the location names from scraped files to match the neighborhoods file.
    """

    name = re.sub(r'^\s*v.z. Boyana\s*$', 'vilna zona Boyana', name)
    name = re.sub(r'^\s*v.z. Gorna banya\s*$', 'vilna zona Gorna banya', name)
    name = (name
    .replace('Sofia,', '')
    .replace('m-t', '')
    .replace('v.z.', '') # clean v.z. locations that are not included in the listings
    .replace('zh.gr.', '')
    .replace('Musagenitsa', 'Mladost 1') # Include not mentioned neighborhood into a larger mentioned neighborhood
    .replace('Doktorski pametnik', 'Tsentar')
    .replace('Oborishte', 'Tsentar')
    .replace('Eksperimentalen', 'Gorublyane')
    .replace('Kambanite', 'Malinova dolina')
    .replace('Malashevtsi', 'NPZ Hadzhi Dimitar')
    .replace('Meditsinska akademiya', 'Kriva reka')
    .replace('Pavlovo', 'Bakston')
    .replace('Reduta', 'Poduyane')
    .replace('Tolstoy', 'Lev Tolstoy') # Match the namings
    .replace('Triagalnika', 'Triagalnika - Nadezhda')
    .replace('Botunets 2', 'Botunets'))

    # Match special namings that require regex to avoid breaking the data
    name = re.sub(r'\bSPZ\b', 'Skladovo-Promishlena Zona', name) 
    name = re.sub(r'\bPZ\b', 'Promishlena Zona', name)
    name = re.sub(r'^\s*Levski\s*$', 'Vasil Levski', name)
    name = re.sub(r'^\s*Lyulin\s*$', 'vilna zona Lyulin', name)
    name = re.sub(r'^\s*Ovcha kupel\s*$', 'Ovcha kupel 1', name)
    name = re.sub(r'^\s*Slatina\s*$', 'Hristo Smirnenski', name)
    return name.strip().lower()

# Applies the standardize function
sofia_for_rent_scraped['location_standardized'] = [standardize_locations(loc) for loc in sofia_for_rent_scraped['location']]
sofia_for_sale_scraped['location_standardized'] = [standardize_locations(loc) for loc in sofia_for_sale_scraped['location']]

In [24]:
# Transcribes neighborhood names to latin, standardizes the latin names to match the scraped files
neighborhoods_standardized = neighborhoods.copy()

neighborhoods_standardized['kvname_latin_standardized'] = [
    (to_latin_bg(name.removeprefix(pref).strip().lower())
     .replace('7-mi  11-ti kilometar', '7-mi 11-ti kilometar')
     .replace('iii chast', '3 chast')
     .replace('malinova dolina- gerena', 'malinova dolina - gerena')
     .replace('simeonovo-dragalevtsi', 'simeonovo - dragalevtsi')
     .replace('razsadnika-konjovitsa', 'razsadnika')
     .replace('triagalnika-nadezhda', 'triagalnika - nadezhda')
     .replace('vrana-lozen triagalnik', 'vrana - lozen'))
    if isinstance(pref, str) else to_latin_bg(name)
    for name, pref in zip(neighborhoods_standardized['kvname'], neighborhoods_standardized['prefname'])
]

# Fixes broken data from the original file
neighborhoods_standardized.loc[neighborhoods_standardized['object_id'] == 131, 'kvname_latin_standardized'] = 'skladovo-promishlena zona slatina'
neighborhoods_standardized.loc[neighborhoods_standardized['object_id'] == 33, 'kvname_latin_standardized'] = 'skladovo-promishlena zona moderno predgradie'
# Accounts for separate v.z. locations
neighborhoods_standardized.loc[neighborhoods_standardized['object_id'] == 148, 'kvname_latin_standardized'] = 'vilna zona gorna banya'
neighborhoods_standardized.loc[neighborhoods_standardized['object_id'] == 184, 'kvname_latin_standardized'] = 'vilna zona boyana'
neighborhoods_standardized.loc[neighborhoods_standardized['object_id'] == 207, 'kvname_latin_standardized'] = 'vilna zona malinova dolina' 
neighborhoods_standardized.loc[neighborhoods_standardized['object_id'] == 173, 'kvname_latin_standardized'] = 'vilna zona lyulin'

# Dealing with duplicate neighborhood names
neighborhoods_standardized.drop(neighborhoods_standardized[neighborhoods_standardized['object_id'].isin([268, 244, # 'В.З. МАНАСТИРА' duplicate, not in the locations
                                                                  278, 57, # 'КВ. ИЗГРЕВ' duplicate
                                                                  310, # 'ЗАПАДЕН ПАРК' a park, not a neighborhood
                                                                  118, 135, # 'КВ. ГРАДОМАН', 'В.З. ГРАДОМАН' not in the locations
                                                                  218, 253, # 'В.З. ТЕРАСИТЕ' duplicate, not in the locations
                                                                  213, 259 # 'В.З. УМНИ ДОЛ', 'М. УМНИ ДОЛ' not in the locations
                                                                  ])].index, inplace=True)

# Combines the geometry of the remaining neighborhoods_standardized that share the same standardized latin name
neighborhoods_standardized = neighborhoods_standardized.dissolve('kvname_latin_standardized')


In [25]:
# Converts sofia_for_rent_scraped and sofia_for_sale_scraped variables from DataFrame into GeoDataFrame
# Puts standardized location column next to the original location column

sofia_for_rent_scraped['geometry'] = sofia_for_rent_scraped['location_standardized'].map(neighborhoods_standardized['geometry'])
sofia_for_rent_scraped = gpd.GeoDataFrame(sofia_for_rent_scraped, geometry='geometry', crs=neighborhoods_standardized.crs)
sofia_for_rent_scraped = sofia_for_rent_scraped[[
       'url', 'ad_id', 'listing_type', 'location', 'location_standardized', 'price_eur', 'area_sqm',
       'price_per_sqm_eur', 'floor', 'central_heating', 'gas',
       'description_bulgarian', 'features', 'views', 'saves',
       'ad_published_date', 'agency', 'date_scraped',
       'geometry']]

sofia_for_sale_scraped['geometry'] = sofia_for_sale_scraped['location_standardized'].map(neighborhoods_standardized['geometry'])
sofia_for_sale_scraped = gpd.GeoDataFrame(sofia_for_sale_scraped, geometry='geometry', crs=neighborhoods_standardized.crs)
sofia_for_sale_scraped = sofia_for_sale_scraped[[
       'url', 'ad_id', 'listing_type', 'location', 'location_standardized', 'price_eur', 'area_sqm',
       'price_per_sqm_eur', 'floor', 'central_heating', 'gas',
       'construction_type', 'construction_year', 'description_bulgarian',
       'features', 'views', 'saves', 'ad_published_date', 'agency',
       'date_scraped', 'geometry']]

In [26]:
neighborhoods_standardized_reset = neighborhoods_standardized.reset_index() # Resets index for a separte variable not to break data
nb = neighborhoods_standardized_reset[['kvname_latin_standardized', 'geometry']].drop_duplicates().to_crs(32635) # Converts to meter system to ensure matching
dc = district_codes[['obns_lat', 'geometry']].to_crs(32635)

# 1. Calculates the total area of each neighborhood
nb['total_area'] = nb.geometry.area

# 2. Cuts each neighborhood along the district borders
pieces = gpd.overlay(nb, dc, how='intersection')
pieces['piece_area'] = pieces.geometry.area

# 3. Calculates the area of each neighborhood inside each district
per_district = pieces.groupby(['kvname_latin_standardized', 'obns_lat'], as_index=False)['piece_area'].sum()

# 4. Calculates the percentage of each neighborhood in its district
result = nb[['kvname_latin_standardized', 'total_area']].merge(per_district, on='kvname_latin_standardized', how='left')
result['piece_area'] = result['piece_area'].fillna(0)
result['pct_in_one_district'] = (result['piece_area'] / result['total_area'] * 100)

result = (result[['kvname_latin_standardized', 'obns_lat', 'pct_in_one_district']]
          .sort_values('kvname_latin_standardized')
          .reset_index(drop=True))


result[result['kvname_latin_standardized'] != ''].to_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\results\intermediate\pct_of_neighborhood_in_district.csv', index=False)

In [27]:
neighborhoods_standardized.to_parquet(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\results\intermediate\neighborhoods_standardized.parquet')